# 公開用注記（原実験Notebook）

このNotebookは開発時のコードを記録として公開したものです。実験ロジックは変更していません。

**実行上の依存関係：** 元の8D追跡・回避チェックポイントが必要です。公開中の最終12Dモデルから、この移植処理を再現することはできません。

公開版での一連の再実行は確認していません。実行時は参照パスや必要ファイルを確認してください。


# Phase42：上位PPO 8次元→12次元の重み移植（CPU・学習なし）

**目的**：既存の追跡・回避PPOチェックポイントを読み込み、8次元Actor/Criticの最初の層を12次元に拡張する。追加4入力に対する重みはゼロ初期化し、同一8次元観測に対して移植前後のActor出力・Critic出力が一致するか確認する。

**重要な区別**
- 本Phaseでは**姿勢情報は行動に影響しない**（追加4入力の初期重みがゼロ）。姿勢を利用するにはその後の学習が必要。
- 既存の8次元モデルファイルは読み取りのみ。新規モデルを別ファイルとして保存する。
- PPOのOptimizer状態は引き継がない。追加学習時は新規Optimizerを構築する。
- Genesis再実行・GPU・学習は不要。

## 1. 既存チェックポイントを読み込む

In [ ]:
from google.colab import drive
from pathlib import Path
import torch,torch.nn as nn,hashlib,json,csv,zipfile,uuid
import pandas as pd
from IPython.display import display
if not Path("/content/drive/MyDrive").exists():drive.mount("/content/drive")
root=Path("/content/drive/MyDrive/PhysicalAI_Go2_Pursuit")
pursuit_candidates=sorted((root/"training/pursuit").glob("pursuit_v045_*/upper_pursuit_ppo.pt"),key=lambda p:p.stat().st_mtime,reverse=True)
assert pursuit_candidates,"追跡v045のチェックポイントが見つかりません"
paths={"pursuit":pursuit_candidates[0],
       "evasion":root/"training/evasion/evasion_995dbb39/upper_evasion_ppo.pt"}
for name,p in paths.items():
 assert p.is_file(),(name,p)
 print(name,p,"bytes",p.stat().st_size)
def sha256(p):
 h=hashlib.sha256()
 with p.open("rb") as f:
  for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)
 return h.hexdigest()
before={name:sha256(p) for name,p in paths.items()}
print("既存モデルのSHA256",before)


## 2. 学習時と同じネットワーク定義

Actor: 8→128→128→2、Critic: 8→128→128→1。移植後は入力のみ12に変更します。

In [ ]:
class ActorCritic(nn.Module):
 def __init__(self,dim):
  super().__init__()
  self.actor=nn.Sequential(nn.Linear(dim,128),nn.Tanh(),nn.Linear(128,128),nn.Tanh(),nn.Linear(128,2))
  self.critic=nn.Sequential(nn.Linear(dim,128),nn.Tanh(),nn.Linear(128,128),nn.Tanh(),nn.Linear(128,1))
  self.log_std=nn.Parameter(torch.tensor([-0.5,-0.5]))
 def commands(self,raw,max_forward):
  return torch.stack((torch.sigmoid(raw[:,0])*max_forward,torch.tanh(raw[:,1])),dim=1)
def transfer(old,new):
 old_state=old.state_dict()
 new_state=new.state_dict()
 assert set(old_state)==set(new_state)
 for k,v in old_state.items():
  if k in ("actor.0.weight","critic.0.weight"):
   assert v.shape==(128,8) and new_state[k].shape==(128,12)
   new_state[k][:,:8]=v.clone()
   new_state[k][:,8:]=0
  else:
   assert v.shape==new_state[k].shape,(k,v.shape,new_state[k].shape)
   new_state[k]=v.clone()
 new.load_state_dict(new_state,strict=True)
 assert torch.count_nonzero(new.actor[0].weight[:,8:]).item()==0
 assert torch.count_nonzero(new.critic[0].weight[:,8:]).item()==0
 return new


## 3. 追跡・回避モデルをそれぞれ移植して出力一致を検証

In [ ]:
torch.manual_seed(42)
N=512
obs8=torch.randn(N,8,dtype=torch.float32)
# 姿勢は非ゼロ値でテスト。roll/pitchはrad、機体角速度はrad/s
extra=torch.randn(N,4,dtype=torch.float32)*torch.tensor([.25,.25,1.5,1.5])
obs12=torch.cat((obs8,extra),dim=1)
assert obs12.shape==(N,12)
results=[]
models={}
for name,p in paths.items():
 ck=torch.load(p,map_location="cpu",weights_only=False)
 assert isinstance(ck,dict) and "actor_critic" in ck
 old=ActorCritic(8).eval()
 old.load_state_dict(ck["actor_critic"],strict=True)
 new=transfer(old,ActorCritic(12)).eval()
 with torch.no_grad():
  raw_old=old.actor(obs8);raw_new=new.actor(obs12)
  val_old=old.critic(obs8);val_new=new.critic(obs12)
  max_forward=.45 if name=="pursuit" else .30
  cmd_old=old.commands(raw_old,max_forward)
  cmd_new=new.commands(raw_new,max_forward)
  actor_diff=float((raw_old-raw_new).abs().max())
  critic_diff=float((val_old-val_new).abs().max())
  command_diff=float((cmd_old-cmd_new).abs().max())
 assert actor_diff<1e-5 and critic_diff<1e-5 and command_diff<1e-5,(name,actor_diff,critic_diff,command_diff)
 assert torch.equal(old.log_std,new.log_std)
 results.append({"model":name,"samples":N,"max_actor_raw_abs_diff":actor_diff,
                 "max_critic_abs_diff":critic_diff,"max_command_abs_diff":command_diff,
                 "zero_initialized_extra_actor_weights":int(torch.count_nonzero(new.actor[0].weight[:,8:]))==0,
                 "zero_initialized_extra_critic_weights":int(torch.count_nonzero(new.critic[0].weight[:,8:]))==0})
 models[name]=new
display(pd.DataFrame(results))
print("PASS: 追跡・回避ともActor/Critic/指令の出力一致")


## 4. 新しいチェックポイントを別名で保存し、原本不変を確認

移植済みファイルは**学習済み12次元PPOではなく、入力拡張済みの初期モデル**です。再学習の成果と混同しないでください。

In [ ]:
out_root=root/"training/upper12_transfer"
run_dir=out_root/("phase42_"+uuid.uuid4().hex[:8])
run_dir.mkdir(parents=True,exist_ok=False)
output_paths={}
for name,model in models.items():
 target=run_dir/("upper_"+name+"_12d_initialized.pt")
 torch.save({"actor_critic":model.state_dict(),"config":{
  "input_dim":12,"output_dim":2,"source_checkpoint":str(paths[name]),
  "source_sha256":before[name],"transfer":"copy first 8 input columns; zero last 4 columns",
  "trained_on_12d":False,"optimizer_transferred":False}},target)
 output_paths[name]=target
for name,p in paths.items():
 assert sha256(p)==before[name],f"原本SHAが変化しました: {name}"
print("既存チェックポイント不変を確認")
for name,p in output_paths.items():print(name,p,p.stat().st_size,"bytes")


## 5. 保存した12次元チェックポイントを再読込して検証・結果ZIP化

In [ ]:
for name,p in output_paths.items():
 ck=torch.load(p,map_location="cpu",weights_only=False)
 test=ActorCritic(12)
 test.load_state_dict(ck["actor_critic"],strict=True)
 assert ck["config"]["trained_on_12d"] is False
 assert torch.count_nonzero(test.actor[0].weight[:,8:])==0
 assert torch.count_nonzero(test.critic[0].weight[:,8:])==0
pd.DataFrame(results).to_csv(run_dir/"phase42_transfer_verification.csv",index=False,encoding="utf-8-sig")
manifest={"phase":42,"training":False,"genesis_executed":False,"input_dim_before":8,
 "input_dim_after":12,"extra_features":["roll_rad","pitch_rad","body_omega_x_rad_s","body_omega_y_rad_s"],
 "original_sha256":before,"new_checkpoints":{k:str(v) for k,v in output_paths.items()},
 "limitations":["12D weights are initialized, not trained","Optimizer state not transferred",
 "Zero extra weights mean posture inputs do not initially affect actions",
 "Actor/Critic parity on random CPU observations does not prove Genesis stability"]}
(run_dir/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
archive=Path("/content/phase42_upper_ppo_transfer_results.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
 for name in ("phase42_transfer_verification.csv","manifest.json"):
  z.write(run_dir/name,name)
print("検証結果ZIP:",archive)
from google.colab import files
files.download(str(archive))
